# Distanzen, Gewichtung und KNN-Regression

KNN kann auch Zahlenwerte vorhersagen: ungewichtet als Mittelwert, distanzgewichtet mit stärkerem Einfluss naher Punkte. Anschließend vergleichen wir euklidische und Mahalanobis-Distanz.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score
from sklearn.model_selection import GridSearchCV,train_test_split
from sklearn.neighbors import KNeighborsRegressor,KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

rng=np.random.default_rng(42)
X=np.linspace(-3,3,450).reshape(-1,1)
y=1.4*np.sin(1.7*X[:,0])+.25*X[:,0]**2+rng.normal(0,.3,len(X))
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.25,random_state=42)
raster=np.linspace(-3,3,600).reshape(-1,1)
modelle={
 "k=3 uniform":KNeighborsRegressor(3,weights="uniform"),
 "k=15 uniform":KNeighborsRegressor(15,weights="uniform"),
 "k=15 distance":KNeighborsRegressor(15,weights="distance")}
fig,ax=plt.subplots(1,3,figsize=(15,4),sharey=True); rows=[]
for a,(name,m) in zip(ax,modelle.items()):
 m.fit(X_train,y_train); p=m.predict(X_test); rows.append({"Modell":name,"MAE":mean_absolute_error(y_test,p),"RMSE":mean_squared_error(y_test,p)**.5,"R2":r2_score(y_test,p)}); a.scatter(X_train[:,0],y_train,s=8,alpha=.25); a.plot(raster[:,0],m.predict(raster),color="tomato"); a.set_title(name)
plt.show(); display(pd.DataFrame(rows).set_index("Modell").round(3))

## Kleine Suche für Regression

Die Parameter entsprechen der Klassifikation. `weights="distance"` berechnet einen gewichteten Mittelwert. Für `metric="minkowski"` bedeutet `p=1` Manhattan- und `p=2` euklidische Distanz.

In [ ]:
pipe=Pipeline([("scale",StandardScaler()),("knn",KNeighborsRegressor())])
grid={"knn__n_neighbors":[3,5,9,15,25,40],"knn__weights":["uniform","distance"],"knn__p":[1,2]}
suche=GridSearchCV(pipe,grid,scoring="neg_root_mean_squared_error",cv=5,n_jobs=-1).fit(X_train,y_train)
print("Beste Parameter:",suche.best_params_,"CV-RMSE:",round(-suche.best_score_,3))
p=suche.predict(X_test); print("Test-RMSE:",round(mean_squared_error(y_test,p)**.5,3),"R2:",round(r2_score(y_test,p),3))

## Euklidisch und Mahalanobis

Euklidisch misst Luftlinie. Mahalanobis verwendet die inverse Kovarianzmatrix und berücksichtigt damit Streuungsrichtungen und Korrelationen. Sie ist sinnvoll, wenn diese Struktur stabil geschätzt werden kann; bei vielen Merkmalen oder wenig Daten kann die Kovarianzmatrix instabil werden.

In [ ]:
# Zwei stark korrelierte Merkmale
n=500; a=rng.normal(size=n); b=a+rng.normal(0,.22,n); Xc=np.c_[a,b]; yc=(a+b+rng.normal(0,.45,n)>0).astype(int)
Xtr,Xte,ytr,yte=train_test_split(Xc,yc,test_size=.25,random_state=42,stratify=yc)
sc=StandardScaler().fit(Xtr); Ztr=sc.transform(Xtr); Zte=sc.transform(Xte)
VI=np.linalg.inv(np.cov(Ztr,rowvar=False))
euk=KNeighborsClassifier(n_neighbors=11,metric="euclidean").fit(Ztr,ytr)
mah=KNeighborsClassifier(n_neighbors=11,metric="mahalanobis",metric_params={"VI":VI}).fit(Ztr,ytr)
print("Euklidisch Accuracy:",round(euk.score(Zte,yte),3)); print("Mahalanobis Accuracy:",round(mah.score(Zte,yte),3))

x,z=Zte[0],Zte[1]
de=np.sqrt(np.sum((x-z)**2)); dm=np.sqrt((x-z)@VI@(x-z))
print("Dieselben zwei Punkte - euklidisch:",round(de,3),"Mahalanobis:",round(dm,3))

**Praxis:** Starte meist mit `StandardScaler` plus euklidischer/Manhattan-Distanz. Mahalanobis ist eine begründete Alternative bei korrelierten Merkmalen, aber kein automatischer Gewinner. KNN-Vorhersagen werden bei großen Trainingsmengen teuer, weil viele Distanzen berechnet werden müssen.